# Pruebas de mejora de `.geo` y `.sif` en Kabre

Este notebook está pensado para correr en **Kabre**, no como runner local del repo en Windows.
Sigue la lógica de `GeneticLab/notebooks/genes_moea_initial_trials.ipynb`: usa
`PROJECT = /work/jmorera/Genes`, deja resultados en `GeneticLab/experiments/<RUN_LABEL>`,
arranca con `DRY_RUN = True`, y llama los scripts existentes de `genes`.

Compara dos familias:

- `annular_rotor`: modelo actual, con imanes internos anulares que abrazan el rotor.
- `central_pill`: variante tipo Rubén/imagen, con imanes internos sólidos tipo pastilla dentro del impulsor.

La idea de mallado: dejar fino imanes internos y bobina, y mantener capas de fondo fijas para no meter variación numérica innecesaria entre simulaciones.

## 0. Configuración

En Kabre dejá `PROJECT = Path('/work/jmorera/Genes')`. Si estás leyendo esto fuera de Kabre, la celda no debe intentar descubrir un repo local: solo va a avisar que la ruta no existe.

Flujo sugerido:

1. `DRY_RUN = True`: revisar rutas y comandos.
2. Escribir templates por variante.
3. Correr preflight Gmsh/ElmerGrid para corregir `ExternalBC`.
4. Escribir casos con `genes/10_poblacion/write_population_cases.py`.
5. Correr barridos pequeños con `genes/20_ejecucion/sweep_mpi.py`.
6. Graficar resultados y decidir qué geometría pasa al genético.

In [ ]:
from pathlib import Path
from datetime import datetime
import importlib.util
import json
import os
import re
import subprocess
import sys

REQUIRED_IMPORTS = {"numpy": "numpy", "pandas": "pandas", "matplotlib": "matplotlib"}
INSTALL_MISSING = False

missing = [pkg for pkg, module in REQUIRED_IMPORTS.items() if importlib.util.find_spec(module) is None]
if missing:
    print("Faltan paquetes:", missing)
    if INSTALL_MISSING:
        subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
    else:
        raise ImportError("Instalá los paquetes faltantes o cambiá INSTALL_MISSING = True.")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

plt.rcParams.update({"figure.figsize": (9, 5), "axes.grid": True, "grid.alpha": 0.25})

In [ ]:
# Configuraci?n principal.
# Si re-ejecut?s esta celda, por defecto conserva RUN_LABEL para no partir el experimento
# entre carpetas distintas. Para crear uno nuevo, pon? FORCE_NEW_EXPERIMENT = True una vez.

DRY_RUN = globals().get("DRY_RUN", True)
FORCE_NEW_EXPERIMENT = False
RESUME_RUN_LABEL = ""  # ejemplo: "geo_sif_trials_20260501_190910" si quer?s retomar uno existente

PROJECT = Path(os.environ.get("KABRE_PROJECT", "/work/jmorera/Genes")).resolve()
LAB_ROOT = PROJECT / "GeneticLab"
GENES_ROOT = PROJECT / "genes"
DOCS_ROOT = PROJECT / "docs"
TEMPLATE_DIR = GENES_ROOT / "00_templates_simulacion"

if RESUME_RUN_LABEL:
    RUN_LABEL = RESUME_RUN_LABEL
elif FORCE_NEW_EXPERIMENT or "RUN_LABEL" not in globals():
    RUN_LABEL = "geo_sif_trials_" + datetime.now().strftime("%Y%m%d_%H%M%S")

EXPERIMENT_ROOT = LAB_ROOT / "experiments" / RUN_LABEL
TEMPLATES_ROOT = EXPERIMENT_ROOT / "templates"
POP_ROOT = EXPERIMENT_ROOT / "results" / "01_population"
CASES_ROOT = EXPERIMENT_ROOT / "cases"
RUNS_ROOT = EXPERIMENT_ROOT / "runs"
REPORTS_ROOT = EXPERIMENT_ROOT / "reports"

WRITE_CASES_PY = GENES_ROOT / "10_poblacion" / "write_population_cases.py"
SWEEP_MPI_PY = GENES_ROOT / "20_ejecucion" / "sweep_mpi.py"
COLLECT_RUNS_PY = GENES_ROOT / "30_postproceso" / "genes_slurm_collect_run_summary.py"

BASE_GEO = TEMPLATE_DIR / "StepsHTX.geo"
BASE_SIF = TEMPLATE_DIR / "P1low.sif"
BASE_DEF = TEMPLATE_DIR / "HMB_circuit.definition"

if PROJECT.exists():
    for folder in [TEMPLATES_ROOT, POP_ROOT, CASES_ROOT, RUNS_ROOT, REPORTS_ROOT]:
        folder.mkdir(parents=True, exist_ok=True)
else:
    print("PROJECT no existe en este kernel:", PROJECT)
    print("Esto est? bien si solo est?s leyendo el notebook fuera de Kabre.")

print("PROJECT:", PROJECT)
print("LAB_ROOT:", LAB_ROOT)
print("EXPERIMENT_ROOT:", EXPERIMENT_ROOT)
print("RUN_LABEL:", RUN_LABEL)
print("DRY_RUN:", DRY_RUN)
for path in [BASE_GEO, BASE_SIF, BASE_DEF, WRITE_CASES_PY, SWEEP_MPI_PY, COLLECT_RUNS_PY]:
    print(path, "->", path.exists())

In [ ]:
MODULE_PREAMBLE = """
module purge
module load elmerfem/9.0
module load gmsh/4.15.0
module load mpich/4.1.1
module load lapack/3.12.1
export LD_LIBRARY_PATH=/work/jmorera/compat_gfortran3:$LD_LIBRARY_PATH
export OMP_NUM_THREADS=1
export OPENBLAS_NUM_THREADS=1
export MKL_NUM_THREADS=1
export NUMEXPR_NUM_THREADS=1
export HYDRA_LAUNCHER=fork
""".strip()

def sh(cmd: str, cwd: Path | None = None, execute: bool = True, use_modules: bool = False) -> str:
    full_cmd = f"{MODULE_PREAMBLE}\n{cmd}" if use_modules else cmd
    if not execute:
        return "DRY_RUN command:\n" + full_cmd
    proc = subprocess.run(["bash", "-lc", full_cmd], cwd=str(cwd) if cwd else None, capture_output=True, text=True)
    chunks = []
    if proc.stdout:
        chunks.append("STDOUT:\n" + proc.stdout)
    if proc.stderr:
        chunks.append("STDERR:\n" + proc.stderr)
    if proc.returncode != 0:
        chunks.append(f"[returncode={proc.returncode}]")
    return "\n".join(chunks)

def should_execute(action_name: str, flag: bool) -> bool:
    execute = bool(flag) and not bool(DRY_RUN)
    print(f"{action_name}: execute={execute} | {action_name}={flag} | DRY_RUN={DRY_RUN}")
    return execute

def experiment_root_for_record(record: dict) -> Path:
    # record['template_dir'] = .../GeneticLab/experiments/<RUN_LABEL>/templates/<scenario>
    # Esto evita mezclar templates de una corrida con cases/manifests de otra si re-ejecut?s config.
    template_dir = Path(record["template_dir"])
    return template_dir.parents[1]

## 1. Referencia de la tesis de Rubén

La celda busca PDFs bajo `docs/` y extrae fragmentos con palabras clave. No decide la geometría automáticamente; sirve para copiar dimensiones o criterios al bloque de parámetros.

In [ ]:
pdf_paths = sorted(DOCS_ROOT.rglob("*.pdf")) if DOCS_ROOT.exists() else []
print("PDFs encontrados:")
for i, path in enumerate(pdf_paths):
    print(f"  [{i}] {path.relative_to(PROJECT)}")

SEARCH_TERMS = ["impulsor", "bobina", "iman", "imán", "permanente", "rotor", "pastilla", "magnet"]

def extract_pdf_snippets(pdf_path: Path, terms=SEARCH_TERMS, max_pages: int = 20) -> pd.DataFrame:
    try:
        from pypdf import PdfReader
    except ImportError:
        try:
            from PyPDF2 import PdfReader
        except ImportError:
            print("No está pypdf/PyPDF2. Instalalo si querés extraer texto del PDF.")
            return pd.DataFrame(columns=["page", "hits", "snippet"])
    reader = PdfReader(str(pdf_path))
    rows = []
    for page_idx, page in enumerate(reader.pages[:max_pages]):
        text = page.extract_text() or ""
        clean = " ".join(text.split())
        lower = clean.lower()
        hits = [term for term in terms if term.lower() in lower]
        if hits:
            rows.append({"page": page_idx + 1, "hits": ", ".join(hits), "snippet": clean[:900]})
    return pd.DataFrame(rows)

thesis_notes = extract_pdf_snippets(pdf_paths[0]) if pdf_paths else pd.DataFrame()
display(thesis_notes)

## 2. Parámetros y perfiles de malla

`write_population_cases.py` hoy actualiza principalmente bobina y circuito. Por eso los cambios estructurales del `.geo` se escriben como **templates por variante**. Si después querés optimizar `r_core`, `h0`, `r1/r2` o `r3/r4`, el siguiente paso es extender ese writer.

In [ ]:
BASE_PARAMS_MM = {
    "dx_mm": 0.0, "dy_mm": 0.0, "dz_mm": 0.0,
    "r1_mm": 6.5, "r2_mm": 11.5, "r3_mm": 12.7, "r4_mm": 25.4,
    "h0_mm": 3.2, "h1_mm": 3.4, "gap_z_mm": 1.0,
    "rb1_mm": 12.7, "rb2_mm": 18.7, "hb_mm": 12.7, "gap_pc_mm": 4.0,
    "r_core_mm": 5.2, "rotor_body_r_mm": 11.5,
    "I_eval_A": 3.0, "Ni": 700, "R_Coil1_ohm": 9.0,
}

def mm(value: float) -> float:
    return float(value) * 1e-3

def with_derived(params: dict) -> dict:
    p = dict(params)
    p["H_pmb_mm"] = 2 * p["h0_mm"] + p["gap_z_mm"]
    p["H_total_mm"] = p["H_pmb_mm"] + p["gap_pc_mm"] + p["hb_mm"]
    p["z_pmb_bottom_mm"] = -p["H_total_mm"] / 2
    p["z_mi1_mm"] = p["z_pmb_bottom_mm"]
    p["z_mi2_mm"] = p["z_mi1_mm"] + p["h0_mm"] + p["gap_z_mm"]
    p["z_me1_mm"] = p["z_mi1_mm"] + (p["h0_mm"] - p["h1_mm"]) / 2
    p["z_me2_mm"] = p["z_me1_mm"] + p["h0_mm"] + p["gap_z_mm"]
    p["z_coil_mm"] = p["z_pmb_bottom_mm"] + p["H_pmb_mm"] + p["gap_pc_mm"]
    p["Ae_Coil1_m2"] = mm(p["rb2_mm"] - p["rb1_mm"]) * mm(p["hb_mm"])
    return p

MESH_PROFILES = {
    "coarse_debug": {
        "air_R_mm": 45.0, "air_H_mm": 52.0,
        "lc_inner_mm": 0.80, "lc_coil_mm": 0.90, "lc_near_mm": 1.40, "lc_mid_mm": 2.20, "lc_far_mm": 4.00,
        "inner_margin_r_mm": 3.0, "inner_margin_z_mm": 2.0, "coil_margin_r_mm": 2.0, "coil_margin_z_mm": 2.0,
        "fixed_near_R_mm": 26.0, "fixed_near_Zmin_mm": -24.0, "fixed_near_Zmax_mm": 18.0,
        "fixed_mid_R_mm": 34.0, "fixed_mid_Zmin_mm": -28.0, "fixed_mid_Zmax_mm": 24.0,
    },
    "balanced_fixed_layers": {
        "air_R_mm": 45.0, "air_H_mm": 52.0,
        "lc_inner_mm": 0.40, "lc_coil_mm": 0.55, "lc_near_mm": 0.90, "lc_mid_mm": 1.80, "lc_far_mm": 3.20,
        "inner_margin_r_mm": 2.5, "inner_margin_z_mm": 1.5, "coil_margin_r_mm": 2.0, "coil_margin_z_mm": 2.0,
        "fixed_near_R_mm": 26.0, "fixed_near_Zmin_mm": -24.0, "fixed_near_Zmax_mm": 18.0,
        "fixed_mid_R_mm": 36.0, "fixed_mid_Zmin_mm": -30.0, "fixed_mid_Zmax_mm": 26.0,
    },
    "fine_inner_coil": {
        "air_R_mm": 50.0, "air_H_mm": 60.0,
        "lc_inner_mm": 0.28, "lc_coil_mm": 0.40, "lc_near_mm": 0.75, "lc_mid_mm": 1.50, "lc_far_mm": 3.00,
        "inner_margin_r_mm": 2.5, "inner_margin_z_mm": 1.5, "coil_margin_r_mm": 2.0, "coil_margin_z_mm": 2.0,
        "fixed_near_R_mm": 28.0, "fixed_near_Zmin_mm": -26.0, "fixed_near_Zmax_mm": 20.0,
        "fixed_mid_R_mm": 40.0, "fixed_mid_Zmin_mm": -34.0, "fixed_mid_Zmax_mm": 30.0,
    },
}
display(pd.DataFrame(MESH_PROFILES).T)

In [ ]:
def add_rect(ax, r0, r1, z0, h, label, color, alpha=0.75, hatch=None):
    ax.add_patch(Rectangle((r0, z0), r1 - r0, h, facecolor=color, edgecolor="black", alpha=alpha, hatch=hatch, label=label))

def plot_rz_geometry(params: dict, variant: str, mesh_profile: str = "balanced_fixed_layers", ax=None):
    p = with_derived(params)
    mesh = MESH_PROFILES[mesh_profile]
    colors = {"inner": "#4C78A8", "outer": "#F58518", "coil": "#54A24B", "rotor": "#999999", "layer": "#B279A2"}
    if ax is None:
        _, ax = plt.subplots(figsize=(8, 7))
    add_rect(ax, 0, mesh["fixed_mid_R_mm"], mesh["fixed_mid_Zmin_mm"], mesh["fixed_mid_Zmax_mm"] - mesh["fixed_mid_Zmin_mm"], "fixed mid", colors["layer"], 0.08)
    add_rect(ax, 0, mesh["fixed_near_R_mm"], mesh["fixed_near_Zmin_mm"], mesh["fixed_near_Zmax_mm"] - mesh["fixed_near_Zmin_mm"], "fixed near", colors["layer"], 0.12)
    add_rect(ax, 0, p["rotor_body_r_mm"], p["z_pmb_bottom_mm"], p["H_pmb_mm"], "rotor envelope", colors["rotor"], 0.12, hatch="//")
    add_rect(ax, p["r3_mm"], p["r4_mm"], p["z_me1_mm"], p["h1_mm"], "outer magnets", colors["outer"], 0.65)
    add_rect(ax, p["r3_mm"], p["r4_mm"], p["z_me2_mm"], p["h1_mm"], "outer magnets", colors["outer"], 0.65)
    if variant == "annular_rotor":
        add_rect(ax, p["r1_mm"], p["r2_mm"], p["z_mi1_mm"], p["h0_mm"], "inner annular", colors["inner"], 0.75)
        add_rect(ax, p["r1_mm"], p["r2_mm"], p["z_mi2_mm"], p["h0_mm"], "inner annular", colors["inner"], 0.75)
    else:
        add_rect(ax, 0, p["r_core_mm"], p["z_mi1_mm"], p["h0_mm"], "inner pill", colors["inner"], 0.75)
        add_rect(ax, 0, p["r_core_mm"], p["z_mi2_mm"], p["h0_mm"], "inner pill", colors["inner"], 0.75)
    add_rect(ax, p["rb1_mm"], p["rb2_mm"], p["z_coil_mm"], p["hb_mm"], "coil", colors["coil"], 0.55, hatch="xx")
    ax.axvline(0, color="black", linewidth=1)
    ax.set_xlabel("radio r [mm]")
    ax.set_ylabel("z [mm]")
    ax.set_title(f"{variant} | {mesh_profile}")
    ax.set_xlim(0, max(mesh["air_R_mm"], p["r4_mm"]) * 1.05)
    ax.set_ylim(-mesh["air_H_mm"] / 2 * 1.05, mesh["air_H_mm"] / 2 * 1.05)
    handles, labels = ax.get_legend_handles_labels()
    unique = dict(zip(labels, handles))
    ax.legend(unique.values(), unique.keys(), loc="upper right")
    return ax

fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)
plot_rz_geometry(BASE_PARAMS_MM, "annular_rotor", ax=axes[0])
plot_rz_geometry({**BASE_PARAMS_MM, "r_core_mm": 5.2}, "central_pill", ax=axes[1])
plt.tight_layout()

## 3. Generar templates por variante

El `.geo` generado conserva `Physical Volume` para los cuerpos `1,3,5,7,20,30`. El `ExternalBC` se corrige luego con preflight, porque las fronteras sí pueden cambiar.

In [ ]:
def fmt_m(value_mm: float) -> str:
    return f"{mm(value_mm):.12g}"

def render_geo(params: dict, variant: str, mesh: dict) -> str:
    p = with_derived(params)
    if variant == "annular_rotor":
        inner = """
Cylinder(5) = {dx, dy, z_mi1+dz,  0,0,h0,  r2, 2*Pi};
Cylinder(6) = {dx, dy, z_mi1+dz,  0,0,h0,  r1, 2*Pi};
MagIntInf[] = BooleanDifference{ Volume{5}; Delete; }{ Volume{6}; Delete; };
InnerMag1 = MagIntInf[0];

Cylinder(7) = {dx, dy, z_mi2+dz,  0,0,h0,  r2, 2*Pi};
Cylinder(8) = {dx, dy, z_mi2+dz,  0,0,h0,  r1, 2*Pi};
MagIntSup[] = BooleanDifference{ Volume{7}; Delete; }{ Volume{8}; Delete; };
InnerMag2 = MagIntSup[0];
""".strip()
        inner_ref_r = p["r2_mm"] + mesh["inner_margin_r_mm"]
    else:
        inner = """
Cylinder(5) = {dx, dy, z_mi1+dz,  0,0,h0,  r_core, 2*Pi};
InnerMag1 = 5;

Cylinder(7) = {dx, dy, z_mi2+dz,  0,0,h0,  r_core, 2*Pi};
InnerMag2 = 7;
""".strip()
        inner_ref_r = p["r_core_mm"] + mesh["inner_margin_r_mm"]

    z_inner_min = p["z_mi1_mm"] - mesh["inner_margin_z_mm"]
    z_inner_max = p["z_mi2_mm"] + p["h0_mm"] + mesh["inner_margin_z_mm"]
    coil_ref_r = p["rb2_mm"] + mesh["coil_margin_r_mm"]
    z_coil_min = p["z_coil_mm"] - mesh["coil_margin_z_mm"]
    z_coil_max = p["z_coil_mm"] + p["hb_mm"] + mesh["coil_margin_z_mm"]

    return f"""
SetFactory("OpenCASCADE");

use_coil = 1;
dx = {fmt_m(p['dx_mm'])};
dy = {fmt_m(p['dy_mm'])};
dz = {fmt_m(p['dz_mm'])};
r1 = {fmt_m(p['r1_mm'])};
r2 = {fmt_m(p['r2_mm'])};
r3 = {fmt_m(p['r3_mm'])};
r4 = {fmt_m(p['r4_mm'])};
r_core = {fmt_m(p['r_core_mm'])};
h0 = {fmt_m(p['h0_mm'])};
h1 = {fmt_m(p['h1_mm'])};
gap_z = {fmt_m(p['gap_z_mm'])};
rb1 = {fmt_m(p['rb1_mm'])};
rb2 = {fmt_m(p['rb2_mm'])};
hb  = {fmt_m(p['hb_mm'])};
gap_pc = {fmt_m(p['gap_pc_mm'])};

H_pmb = 2*h0 + gap_z;
H_total = H_pmb + gap_pc + hb;
z_pmb_bottom = -H_total/2;
air_R = {fmt_m(mesh['air_R_mm'])};
air_H = {fmt_m(mesh['air_H_mm'])};
z_air = -air_H/2;

z_mi1 = z_pmb_bottom;
z_mi2 = z_mi1 + h0 + gap_z;
{inner}

z_me1 = z_mi1 + (h0 - h1)/2;
Cylinder(1) = {{0,0,z_me1, 0,0,h1, r4, 2*Pi}};
Cylinder(2) = {{0,0,z_me1, 0,0,h1, r3, 2*Pi}};
MagExtInf[] = BooleanDifference{{ Volume{{1}}; Delete; }}{{ Volume{{2}}; Delete; }};
OuterMag1 = MagExtInf[0];
z_me2 = z_me1 + h0 + gap_z;
Cylinder(3) = {{0,0,z_me2, 0,0,h1, r4, 2*Pi}};
Cylinder(4) = {{0,0,z_me2, 0,0,h1, r3, 2*Pi}};
MagExtSup[] = BooleanDifference{{ Volume{{3}}; Delete; }}{{ Volume{{4}}; Delete; }};
OuterMag2 = MagExtSup[0];

z_coil = z_pmb_bottom + H_pmb + gap_pc;
Cylinder(20) = {{0,0,z_coil, 0,0,hb, rb2, 2*Pi}};
Cylinder(21) = {{0,0,z_coil, 0,0,hb, rb1, 2*Pi}};
CoilVol[] = BooleanDifference{{ Volume{{20}}; Delete; }}{{ Volume{{21}}; Delete; }};
Coil = CoilVol[0];

Cylinder(30) = {{0,0,z_air, 0,0,air_H, air_R, 2*Pi}};
AirClean[] = BooleanDifference{{ Volume{{30}}; Delete; }}{{ Volume{{OuterMag1, OuterMag2, InnerMag1, InnerMag2, Coil}}; }};
Air = AirClean[0];
Coherence;

Physical Volume("OuterMag1", 1) = {{OuterMag1}};
Physical Volume("OuterMag2", 3) = {{OuterMag2}};
Physical Volume("InnerMag1", 5) = {{InnerMag1}};
Physical Volume("InnerMag2", 7) = {{InnerMag2}};
Physical Volume("Coil", 20) = {{Coil}};
Physical Volume("Air", 30) = {{Air}};

Mesh.Algorithm3D = 10;
Mesh.Smoothing = 8;
Mesh.Optimize = 1;
Mesh.OptimizeNetgen = 1;
Mesh.OptimizeThreshold = 0.45;
lc_inner = {fmt_m(mesh['lc_inner_mm'])};
lc_coil = {fmt_m(mesh['lc_coil_mm'])};
lc_near = {fmt_m(mesh['lc_near_mm'])};
lc_mid = {fmt_m(mesh['lc_mid_mm'])};
lc_far = {fmt_m(mesh['lc_far_mm'])};
Mesh.CharacteristicLengthMin = lc_inner;
Mesh.CharacteristicLengthMax = lc_far;

Field[10] = Box;
Field[10].VIn = lc_inner; Field[10].VOut = lc_far;
Field[10].XMin = -{fmt_m(inner_ref_r)}; Field[10].XMax = {fmt_m(inner_ref_r)};
Field[10].YMin = -{fmt_m(inner_ref_r)}; Field[10].YMax = {fmt_m(inner_ref_r)};
Field[10].ZMin = {fmt_m(z_inner_min)}; Field[10].ZMax = {fmt_m(z_inner_max)};

Field[11] = Box;
Field[11].VIn = lc_coil; Field[11].VOut = lc_far;
Field[11].XMin = -{fmt_m(coil_ref_r)}; Field[11].XMax = {fmt_m(coil_ref_r)};
Field[11].YMin = -{fmt_m(coil_ref_r)}; Field[11].YMax = {fmt_m(coil_ref_r)};
Field[11].ZMin = {fmt_m(z_coil_min)}; Field[11].ZMax = {fmt_m(z_coil_max)};

Field[12] = Box;
Field[12].VIn = lc_near; Field[12].VOut = lc_far;
Field[12].XMin = -{fmt_m(mesh['fixed_near_R_mm'])}; Field[12].XMax = {fmt_m(mesh['fixed_near_R_mm'])};
Field[12].YMin = -{fmt_m(mesh['fixed_near_R_mm'])}; Field[12].YMax = {fmt_m(mesh['fixed_near_R_mm'])};
Field[12].ZMin = {fmt_m(mesh['fixed_near_Zmin_mm'])}; Field[12].ZMax = {fmt_m(mesh['fixed_near_Zmax_mm'])};

Field[13] = Box;
Field[13].VIn = lc_mid; Field[13].VOut = lc_far;
Field[13].XMin = -{fmt_m(mesh['fixed_mid_R_mm'])}; Field[13].XMax = {fmt_m(mesh['fixed_mid_R_mm'])};
Field[13].YMin = -{fmt_m(mesh['fixed_mid_R_mm'])}; Field[13].YMax = {fmt_m(mesh['fixed_mid_R_mm'])};
Field[13].ZMin = {fmt_m(mesh['fixed_mid_Zmin_mm'])}; Field[13].ZMax = {fmt_m(mesh['fixed_mid_Zmax_mm'])};

Field[14] = Min;
Field[14].FieldsList = {{10,11,12,13}};
Background Field = 14;
Mesh.MshFileVersion = 2.2;
Mesh.Binary = 0;
Mesh.SaveParametric = 0;
Mesh.SaveAll = 1;
Mesh 3;
""".strip() + "\n"

In [ ]:
def replace_definition_assignment(text: str, var: str, value: str) -> str:
    pat = re.compile(rf"(?m)^(\s*\$?\s*{re.escape(var)}\s*=\s*)([^\n\r!#;]*)(.*)$")
    if not pat.search(text):
        raise RuntimeError(f"No encontré {var} en .definition")
    return pat.sub(rf"\g<1>{value}\g<3>", text, count=1)

def render_definition(params: dict, base_text: str) -> str:
    p = with_derived(params)
    text = replace_definition_assignment(base_text, "I1", f"{p['I_eval_A']:.12g}")
    text = replace_definition_assignment(text, "N_Coil1", f"{int(p['Ni'])}")
    text = replace_definition_assignment(text, "Ae_Coil1", f"{p['Ae_Coil1_m2']:.12g}")
    return text

def update_external_bc(sif_text: str, boundary_ids: list[int]) -> str:
    ids = " ".join(str(int(x)) for x in boundary_ids)
    updated = re.sub(r"Target Boundaries\(\d+\)\s*=\s*[0-9 ]+", f"Target Boundaries({len(boundary_ids)}) = {ids}", sif_text)
    if updated == sif_text:
        raise ValueError("No encontré Target Boundaries(...) en el SIF")
    return updated

## 4. Plan de experimentos

In [ ]:
SCENARIOS = pd.DataFrame([
    {"name": "A_annular_coarse", "variant": "annular_rotor", "mesh_profile": "coarse_debug", "r_core_mm": np.nan, "reason": "smoke test"},
    {"name": "A_annular_balanced", "variant": "annular_rotor", "mesh_profile": "balanced_fixed_layers", "r_core_mm": np.nan, "reason": "baseline actual"},
    {"name": "B_pill_r5p2_balanced", "variant": "central_pill", "mesh_profile": "balanced_fixed_layers", "r_core_mm": 5.2, "reason": "pastilla tipo imagen"},
    {"name": "B_pill_r6p0_balanced", "variant": "central_pill", "mesh_profile": "balanced_fixed_layers", "r_core_mm": 6.0, "reason": "sensibilidad radio pastilla"},
    {"name": "A_annular_fine", "variant": "annular_rotor", "mesh_profile": "fine_inner_coil", "r_core_mm": np.nan, "reason": "convergencia de malla"},
    {"name": "B_pill_r5p2_fine", "variant": "central_pill", "mesh_profile": "fine_inner_coil", "r_core_mm": 5.2, "reason": "convergencia de malla"},
])
display(SCENARIOS)

In [ ]:
def scenario_params(row: pd.Series) -> dict:
    p = dict(BASE_PARAMS_MM)
    if not pd.isna(row.get("r_core_mm", np.nan)):
        p["r_core_mm"] = float(row["r_core_mm"])
    return with_derived(p)

def write_scenario_template(row: pd.Series) -> dict:
    if not BASE_SIF.exists() or not BASE_DEF.exists():
        raise FileNotFoundError("Faltan templates base en genes/00_templates_simulacion")
    params = scenario_params(row)
    mesh = MESH_PROFILES[row["mesh_profile"]]
    scenario_dir = TEMPLATES_ROOT / row["name"]
    scenario_dir.mkdir(parents=True, exist_ok=True)
    geo_path = scenario_dir / "StepsHTX.geo"
    sif_path = scenario_dir / "P1low.sif"
    def_path = scenario_dir / "HMB_circuit.definition"
    geo_path.write_text(render_geo(params, row["variant"], mesh), encoding="utf-8")
    sif_path.write_text(BASE_SIF.read_text(encoding="utf-8", errors="ignore"), encoding="utf-8")
    def_path.write_text(render_definition(params, BASE_DEF.read_text(encoding="utf-8", errors="ignore")), encoding="utf-8")
    (scenario_dir / "scenario_config.json").write_text(json.dumps({"scenario": row.to_dict(), "params_mm": params, "mesh": mesh}, indent=2), encoding="utf-8")
    return {"scenario": row["name"], "template_dir": scenario_dir, "geo": geo_path, "sif": sif_path, "definition": def_path}

def write_one_row_population(row: pd.Series) -> Path:
    params = scenario_params(row)
    pop = pd.DataFrame([{
        "individual_id": row["name"], "generation": 0, "algorithm": "geo_sif_trial",
        "rb1_mm": params["rb1_mm"], "rb2_mm": params["rb2_mm"], "hb_mm": params["hb_mm"], "gap_pc_mm": params["gap_pc_mm"],
        "I_eval_A": params["I_eval_A"], "Ni": params["Ni"],
        "geometry_variant": row["variant"], "mesh_profile": row["mesh_profile"], "r_core_mm": params["r_core_mm"],
    }])
    out = POP_ROOT / f"{row['name']}_population.csv"
    out.parent.mkdir(parents=True, exist_ok=True)
    pop.to_csv(out, index=False)
    return out

WRITE_TEMPLATES = True
scenario_records = []
if WRITE_TEMPLATES:
    for _, row in SCENARIOS.iterrows():
        record = write_scenario_template(row)
        record["population_csv"] = write_one_row_population(row)
        scenario_records.append(record)
    display(pd.DataFrame([{k: str(v) for k, v in rec.items()} for rec in scenario_records]))

## 5. Preflight y `ExternalBC`

Este paso corre Gmsh + ElmerGrid para cada template, busca superficies 2D externas del aire (`30-exterior`) y actualiza el `P1low.sif` del template. Dejalo en `RUN_PREFLIGHT=False` hasta estar en Kabre y haber revisado los comandos.

In [ ]:
def load_elmer_mesh_maps(mesh_dir: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    elem_body, body_counts = {}, {}
    with (mesh_dir / "mesh.elements").open(encoding="utf-8", errors="ignore") as f:
        for line in f:
            parts = line.split()
            if len(parts) < 3:
                continue
            eid, body = int(parts[0]), int(parts[1])
            elem_body[eid] = body
            body_counts[body] = body_counts.get(body, 0) + 1
    rows = []
    with (mesh_dir / "mesh.boundary").open(encoding="utf-8", errors="ignore") as f:
        for line in f:
            parts = line.split()
            if len(parts) < 6:
                continue
            parent_a = elem_body.get(int(parts[2]), 0)
            parent_b = elem_body.get(int(parts[3]), 0) if int(parts[3]) > 0 else 0
            pair = "-".join(map(str, sorted([parent_a, parent_b]))) if parent_b > 0 else f"{parent_a}-exterior"
            rows.append({"Boundary": int(parts[1]), "ElementType": int(parts[4]), "ParentPairs": pair})
    bodies = pd.DataFrame([{"Body": k, "Elements": v} for k, v in sorted(body_counts.items())])
    boundaries = pd.DataFrame(rows)
    if not boundaries.empty:
        boundaries = boundaries.groupby(["Boundary", "ElementType", "ParentPairs"], as_index=False).size().rename(columns={"size": "Elements"})
    return bodies, boundaries

def external_air_boundary_ids(boundaries: pd.DataFrame, air_body: int = 30) -> list[int]:
    if boundaries.empty:
        return []
    mask = (boundaries["ParentPairs"] == f"{air_body}-exterior") & (boundaries["ElementType"] == 303)
    return sorted(boundaries.loc[mask, "Boundary"].astype(int).unique().tolist())

def preflight_template(template_dir: Path):
    out = template_dir / "preflight"
    cmd = f"""
set -euo pipefail
rm -rf {out}
mkdir -p {out}
gmsh {template_dir / 'StepsHTX.geo'} -3 -format msh2 -save_all -o {out / 'mesh.msh'}
cd {out}
ElmerGrid 14 2 mesh.msh -out mesh
"""
    print(sh(cmd, cwd=PROJECT, execute=should_execute("RUN_PREFLIGHT", RUN_PREFLIGHT), use_modules=True))
    if DRY_RUN or not RUN_PREFLIGHT:
        return None, None, []
    bodies, boundaries = load_elmer_mesh_maps(out / "mesh")
    ids = external_air_boundary_ids(boundaries)
    if not ids:
        raise RuntimeError(f"No encontré superficies externas de Air en {template_dir}")
    sif_path = template_dir / "P1low.sif"
    sif_path.write_text(update_external_bc(sif_path.read_text(encoding="utf-8", errors="ignore"), ids), encoding="utf-8")
    bodies.to_csv(out / "body_counts.csv", index=False)
    boundaries.to_csv(out / "boundary_report.csv", index=False)
    return bodies, boundaries, ids

RUN_PREFLIGHT = not DRY_RUN
if RUN_PREFLIGHT:
    summary = []
    for rec in scenario_records:
        bodies, boundaries, ids = preflight_template(Path(rec["template_dir"]))
        summary.append({"scenario": rec["scenario"], "external_bc": ids, "n_bodies": len(bodies)})
    display(pd.DataFrame(summary))
else:
    print("RUN_PREFLIGHT=False. Cambialo a True en Kabre para corregir ExternalBC.")

## 6. Escribir casos con `write_population_cases.py`

In [ ]:
def write_cases_command(record: dict) -> tuple[str, Path]:
    scenario = record["scenario"]
    exp_root = experiment_root_for_record(record)
    manifest = exp_root / "results" / "01_population" / f"{scenario}_manifest.csv"
    outdir = exp_root / "cases" / scenario
    cmd = f"""
python3 {WRITE_CASES_PY} \
  --population {record['population_csv']} \
  --geo-template {record['geo']} \
  --sif-template {record['sif']} \
  --definition-template {record['definition']} \
  --outdir {outdir} \
  --manifest-out {manifest}
"""
    return cmd, manifest

RUN_WRITE_CASES = not DRY_RUN
manifest_paths = []
execute_write_cases = should_execute("RUN_WRITE_CASES", RUN_WRITE_CASES)

for rec in scenario_records:
    cmd, manifest = write_cases_command(rec)
    print("\n###", rec["scenario"])
    print(sh(cmd, cwd=PROJECT, execute=execute_write_cases, use_modules=False))
    manifest_paths.append(manifest)

if execute_write_cases:
    for manifest in manifest_paths:
        print(manifest, "->", manifest.exists())

## 7. Barrido corto por escenario

In [ ]:
START_MM = -1.0
END_MM = 1.0
STEP_MM = 1.0
INNER_NPROCS = 20
LAUNCHER = "mpirun"
PARTITION_METHOD = "metiskway"
BIND = "core"
RUN_SWEEPS = not DRY_RUN
execute_sweeps = should_execute("RUN_SWEEPS", RUN_SWEEPS)

for rec in scenario_records:
    scenario = rec["scenario"]
    exp_root = experiment_root_for_record(rec)
    case_dir = exp_root / "cases" / scenario / scenario
    outdir = exp_root / "runs" / scenario
    cmd = f"""
python3 {SWEEP_MPI_PY} \
  --geo {case_dir / 'StepsHTX.geo'} \
  --sif {case_dir / 'P1low.sif'} \
  --definition {case_dir / 'HMB_circuit.definition'} \
  --outdir {outdir} \
  --axis dz \
  --start-mm {START_MM} \
  --end-mm {END_MM} \
  --step-mm {STEP_MM} \
  --nprocs {INNER_NPROCS} \
  --partition-method {PARTITION_METHOD} \
  --launcher {LAUNCHER} \
  --bind {BIND}
"""
    print("\n###", scenario)
    print(sh(cmd, cwd=PROJECT, execute=execute_sweeps, use_modules=True))

## 8. Graficar resultados existentes

In [ ]:
def load_sweep_results() -> pd.DataFrame:
    rows = []
    for rec in scenario_records:
        exp_root = experiment_root_for_record(rec)
        csv_path = exp_root / "runs" / rec["scenario"] / "diag_sweep_results.csv"
        if csv_path.exists():
            df = pd.read_csv(csv_path)
            df["scenario"] = rec["scenario"]
            rows.append(df)
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()

results = load_sweep_results()
if results.empty:
    print("No hay diag_sweep_results.csv todav?a.")
else:
    display(results[["scenario", "tag", "dz_m", "W_J", "msh_elements", "status", "note"]].head(30))
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for scenario, group in results.groupby("scenario"):
        x_mm = group["dz_m"] * 1000.0
        axes[0].plot(x_mm, group["W_J"], marker="o", label=scenario)
        axes[1].plot(x_mm, group["msh_elements"], marker="o", label=scenario)
    axes[0].set_xlabel("dz [mm]")
    axes[0].set_ylabel("ElectroMagnetic Field Energy [J]")
    axes[0].legend()
    axes[1].set_xlabel("dz [mm]")
    axes[1].set_ylabel("elementos .msh")
    axes[1].legend()
    plt.tight_layout()

## Criterio de decisión inicial

- Primero correr `coarse_debug` para verificar que la geometría no falle.
- Luego comparar `A_annular_balanced` vs `B_pill_r5p2_balanced` con el mismo barrido `dz`.
- Revisar estabilidad de elementos y estados `OK` del solver.
- Repetir ambos con `fine_inner_coil`. Si la conclusión cambia mucho, falta independencia de malla.
- Si la pastilla central tiene volumen magnético muy distinto, crear una variante de volumen comparable antes de comparar desempeño.